# Flight Price Prediction
The project predicts flight prices using machine learning based on features such as airline, distance, route, journey date, stops, and duration.

## Import Libraries
Importing necessary libraries for data processing, feature engineering, and model building.

In [1]:
from math import radians, sin, cos, sqrt, atan2
from difflib import get_close_matches
import pandas as pd
import numpy as np
import datetime as dt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Route Distance Mapping (Latitude & Longitude)
Latitude and longitude values are used to calculate the distance between source and destination cities.  
This helps capture the effect of route distance on flight ticket prices.

A fallback mechanism is also included to handle incorrect or unmatched city names.

In [2]:
airport_coords = {
    "Delhi": (28.5562, 77.1000),
    "Mumbai": (19.0896, 72.8656),
    "Banglore": (13.1986, 77.7066),
    "Hyderabad": (17.2403, 78.4294),
    "Kolkata": (22.6450, 88.4467),
    "Chennai": (12.9900, 80.1693),
    "Pune": (18.5800, 73.9200),
    "Jaipur": (26.8242, 75.8122),
    "Goa": (15.3800, 73.8300),
    "Ahmedabad": (23.0722, 72.6318)
}

In [3]:
def get_valid_city(city):
    city = city.strip().title()
    if city in airport_coords:
        return city
    matches = get_close_matches(city, airport_coords.keys(), n=1, cutoff=0.6)
    return matches[0] if matches else None

In [4]:
def calculate_distance(city1, city2):
    city1 = get_valid_city(city1)
    city2 = get_valid_city(city2)

    if city1 is None or city2 is None:
        return 850  # fallback avg distance

    lat1, lon1 = airport_coords[city1]
    lat2, lon2 = airport_coords[city2]

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    distance = 6371 * c
    return distance

## Data Loading
Loading the flight dataset containing details such as airline, source, destination, timings, and price.

In [5]:
df = pd.read_excel("Flight_data.xlsx")

## Feature Engineering
Creating meaningful features from raw data such as time, duration, distance, and stops to improve and capture hidden patterns.

### Date & Time Features

In [6]:
df['Date_of_Journey'] = pd.to_datetime(df['Date_of_Journey'], dayfirst=True)
df['Journey_day'] = df['Date_of_Journey'].dt.day
df['Journey_month'] = df['Date_of_Journey'].dt.month
df['Journey_year'] = df['Date_of_Journey'].dt.year
df['Weekday'] = df['Date_of_Journey'].dt.weekday

In [7]:
df['is_weekend'] = df['Weekday'].isin([5,6]).astype(int)

In [8]:
df['is_peak_season'] = df['Journey_month'].isin([3,4,5,6,10,11,12]).astype(int)

In [9]:
today = dt.datetime.today()
df['Days_Before_Booking'] = (df['Date_of_Journey'] - today).dt.days.clip(lower=0)

In [10]:
df['Dep_Time'] = pd.to_datetime(df['Dep_Time'], format='%H:%M', errors='coerce').fillna(pd.Timestamp("00:00"))
df['Dep_hour'] = df['Dep_Time'].dt.hour
df['Dep_min'] = df['Dep_Time'].dt.minute

In [11]:
df['Arrival_Time'] = pd.to_datetime(df['Arrival_Time'], errors='coerce').fillna(pd.Timestamp("00:00"))
df['Arrival_hour'] = df['Arrival_Time'].dt.hour
df['Arrival_min'] = df['Arrival_Time'].dt.minute

C:\Users\354ak\AppData\Local\Temp\ipykernel_12392\4071974817.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Arrival_Time'] = pd.to_datetime(df['Arrival_Time'], errors='coerce').fillna(pd.Timestamp("00:00"))


### Duration & Distance Features

In [12]:
df["Duration_min_total"] = (
    (df["Arrival_hour"]*60 + df["Arrival_min"]) -
    (df["Dep_hour"]*60 + df["Dep_min"])
)
df["Duration_min_total"] = df["Duration_min_total"].apply(lambda x: x + 1440 if x < 0 else x)

In [13]:
df["long_layover"] = (df["Duration_min_total"] > 300).astype(int)

In [14]:
df["Duration_hour"] = df["Duration_min_total"] // 60
df["Duration_min"] = df["Duration_min_total"] % 60

In [15]:
df["Distance_km"] = df.apply(lambda x: calculate_distance(x["Source"], x["Destination"]), axis=1)

### Stops & Route Features

In [16]:
df["Price_route_median"] = df.groupby(["Source","Destination"])["Price"].transform("median")
df["Price"] = (df["Price"] + df["Price_route_median"]) / 2

In [17]:
df["Stops_0"] = (df["Total_Stops"] == "non-stop").astype(int)
df["Stops_1"] = (df["Total_Stops"] == "1 stop").astype(int)
df["Stops_2"] = (df["Total_Stops"] == "2 stops").astype(int)
df["Stops_3"] = (df["Total_Stops"] == "3 stops").astype(int)
df["Stops_4"] = (df["Total_Stops"] == "4 stops").astype(int)
df.drop(["Route", "Date_of_Journey", "Dep_Time", "Arrival_Time"], axis=1, inplace=True)

In [18]:
df["stops1_dist"] = df["Distance_km"] * df["Stops_1"]
df["stops2_dist"] = df["Distance_km"] * df["Stops_2"]

## Data Encoding
Converting categorical values into numerical format.

In [19]:
df_encoded = pd.get_dummies(df, drop_first=True, dtype=int)

## Feature Selection & Target Transformation
Separating features and applying log transformation to improve model performance and handle skewness.

In [20]:
# Apply log transformation to stabilize variance and improve model learning

X = df_encoded.drop("Price", axis=1)
y = df_encoded["Price"]
y_log = np.log1p(y)

## Model Training

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

In [22]:
model = RandomForestRegressor(
    n_estimators=600,
    max_depth=28,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

In [23]:
model.fit(X_train, y_train)

,n_estimators,600
,criterion,'squared_error'
,max_depth,28
,min_samples_split,4
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Model Evaluation
Evaluating model performance using MAE (Mean Absolute Error) and RMSE (Root Mean Squared Error).

In [24]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

## Results
The model achieved the following performance:
- MAE: ~316  
- RMSE: ~667  

This indicates that the model provides reasonably accurate predictions for flight prices.

In [25]:
mae = mean_absolute_error(np.expm1(y_test), y_pred)
rmse = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred))
print("MAE:", mae)
print("RMSE:", rmse)

MAE: 316.5406441925874
RMSE: 667.141143911384


## Prediction Function
Creating a reusable prediction function to estimate flight prices based on user inputs.

In [26]:
def predict_flight_price(
    airline, source, destination, journey_date,
    dep_time, arr_time, stops, additional_info
):

    # 1. Prepare empty row

    model_columns = df_encoded.drop("Price", axis=1).columns.tolist()
    row = pd.DataFrame([[0] * len(model_columns)], columns=model_columns)

    # 2. Stops mapping (local safety)

    stops_map = {
        "non-stop": 0,
        "1 stop": 1,
        "2 stops": 2,
        "3 stops": 3,
        "4 stops": 4
    }

    # 3. Date Features

    journey_dt = dt.datetime.strptime(journey_date, "%d-%m-%Y")

    row.loc[0, "Journey_day"] = journey_dt.day
    row.loc[0, "Journey_month"] = journey_dt.month
    row.loc[0, "Journey_year"] = journey_dt.year
    row.loc[0, "Weekday"] = journey_dt.weekday()
    row.loc[0, "is_weekend"] = 1 if journey_dt.weekday() in [5, 6] else 0
    row.loc[0, "is_peak_season"] = 1 if journey_dt.month in [3,4,5,6,10,11,12] else 0

    today = dt.datetime.today()
    row.loc[0, "Days_Before_Booking"] = max((journey_dt - today).days, 0)

    # 4. Time Features
    
    dep = dt.datetime.strptime(dep_time, "%H:%M")
    arr = dt.datetime.strptime(arr_time, "%H:%M")

    row.loc[0, "Dep_hour"] = dep.hour
    row.loc[0, "Dep_min"] = dep.minute
    row.loc[0, "Arrival_hour"] = arr.hour
    row.loc[0, "Arrival_min"] = arr.minute

    # Duration
    duration = (arr - dep).total_seconds() / 60
    if duration < 0:
        duration += 1440

    row.loc[0, "Duration_min_total"] = duration
    row.loc[0, "Duration_hour"] = int(duration // 60)
    row.loc[0, "Duration_min"] = int(duration % 60)

    # long layover feature
    row.loc[0, "long_layover"] = 1 if duration > 300 else 0

    # 5. Stops

    for col in ["Stops_0", "Stops_1", "Stops_2", "Stops_3", "Stops_4"]:
        if col in model_columns:
            row.loc[0, col] = 0

    # Set correct stop value
    if stops == "non-stop" or stops == 0:
        row.loc[0, "Stops_0"] = 1
    elif stops in ["1 stop", 1]:
        row.loc[0, "Stops_1"] = 1
    elif stops in ["2 stops", 2]:
        row.loc[0, "Stops_2"] = 1
    elif stops in ["3 stops", 3]:
        row.loc[0, "Stops_3"] = 1
    elif stops in ["4 stops", 4]:
        row.loc[0, "Stops_4"] = 1

    # 6. One-hot encoding

    categorical_values = {
        "Airline": airline,
        "Source": source,
        "Destination": destination,
        "Additional_Info": additional_info
    }

    for prefix, value in categorical_values.items():
        col = f"{prefix}_{value}"
        if col in model_columns:
            row.loc[0, col] = 1


    # 7. Distance
 
    if "Distance_km" in model_columns:
        row["Distance_km"] = row["Distance_km"].astype(float)
        row.loc[0, "Distance_km"] = float(calculate_distance(source, destination))

    # 8. Prediction (log reversal)
    
    log_pred = model.predict(row)[0]
    final_price = np.expm1(log_pred)

    return float(final_price)

## Sample Prediction
Example prediction using the trained model with sample input values.

In [44]:
predict_flight_price(
    airline="IndiGo",
    source="Delhi",
    destination="Mumbai",
    journey_date="10-04-2026",
    dep_time="15:00",
    arr_time="19:35",
    stops=1,
    additional_info="1-short layover"
)

6931.914211900172

## Conclusion
This project demonstrates an end-to-end machine learning pipeline for predicting flight prices using real-world data, advanced feature engineering, and a Random Forest model.
Key highlights:
- Applied feature engineering (duration, distance, stops, journey timing) to capture pricing patterns
- Achieved strong performance with MAE ≈ 316 and RMSE ≈ 667  
- Built a reusable prediction function for real-world inputs  
- Successfully translated data into actionable price predictions

Overall, this project highlights the effective use of data analysis, feature engineering, and machine learning techniques to solve a real-world pricing problem.

## Future Improvements
- Try advanced models like XGBoost
- Deploy using Flask/StreamLit